In [1]:
import zipfile

# Path to your ZIP file
zip_path = "/bask/projects/c/chenhp-data-gen/yifansun/project/DPDM/data/processed/cifar10.zip"

# Open the ZIP file and list its contents
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_list = zip_ref.namelist()
    print("Files in the ZIP archive:")
    for file in file_list:
        print(file)

Files in the ZIP archive:
00000/img00000000.png
00000/img00000001.png
00000/img00000002.png
00000/img00000003.png
00000/img00000004.png
00000/img00000005.png
00000/img00000006.png
00000/img00000007.png
00000/img00000008.png
00000/img00000009.png
00000/img00000010.png
00000/img00000011.png
00000/img00000012.png
00000/img00000013.png
00000/img00000014.png
00000/img00000015.png
00000/img00000016.png
00000/img00000017.png
00000/img00000018.png
00000/img00000019.png
00000/img00000020.png
00000/img00000021.png
00000/img00000022.png
00000/img00000023.png
00000/img00000024.png
00000/img00000025.png
00000/img00000026.png
00000/img00000027.png
00000/img00000028.png
00000/img00000029.png
00000/img00000030.png
00000/img00000031.png
00000/img00000032.png
00000/img00000033.png
00000/img00000034.png
00000/img00000035.png
00000/img00000036.png
00000/img00000037.png
00000/img00000038.png
00000/img00000039.png
00000/img00000040.png
00000/img00000041.png
00000/img00000042.png
00000/img00000043.png
00000/

In [4]:
import os
import json
import shutil
from PIL import Image

def transform_dataset(json_path, source_dir, dest_dir, resolution=(256, 256)):
    """
    Transforms the dataset based on the dataset.json file.

    Args:
        json_path (str): Path to the dataset.json file.
        source_dir (str): Directory containing the original images.
        dest_dir (str): Directory to save the transformed dataset.
        resolution (tuple): Desired resolution for the images (width, height).
    """
    # Load the dataset.json file
    with open(json_path, 'r') as f:
        metadata = json.load(f)
    
    # Create destination directory
    os.makedirs(dest_dir, exist_ok=True)
    
    # Process each image
    for image_name, label in metadata['labels']:
        # Create class-specific folder
        class_dir = os.path.join(dest_dir, f"{label}")
        os.makedirs(class_dir, exist_ok=True)
        
        # Load and resize the image
        source_path = os.path.join(source_dir, image_name)
        dest_path = os.path.join(class_dir, os.path.basename(image_name))
        
        with Image.open(source_path) as img:
            img = img.resize(resolution, Image.Resampling.LANCZOS)
            img.save(dest_path)
    
    print(f"Dataset transformed and saved to {dest_dir}")

# Example usage
json_path = "scratch/datasets/cifar10/dataset.json"
source_dir = "scratch/datasets/cifar10"
dest_dir = "scratch/datasets/cifar10formatted"
transform_dataset(json_path, source_dir, dest_dir, resolution=(32,32))

Dataset transformed and saved to scratch/datasets/cifar10formatted


In [1]:
try:
    import torch
    print(torch.cuda.is_available())
except Exception as e:
	print(f"Error: {e}")
	print("Attempting to reinstall torch...")
	# import subprocess
	# import sys
	# subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', 'torch', '-y'])
	# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121'])

	# print("Reinstallation complete. Please restart the kernel and try again.")

False


In [28]:
from PIL import Image
import os
import numpy as np

label_folder = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Label'
new_label_folder = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Label_255'
os.makedirs(new_label_folder, exist_ok=True)
for filename in os.listdir(label_folder):
    if filename.endswith('.png'):
        file_path = os.path.join(label_folder, filename)
        new_file_path = os.path.join(new_label_folder,  filename)
        img = Image.open(file_path)
        img_array = np.array(img)
        
        num_classes = len(np.unique(img_array))
        img_array = img_array * (255 // (num_classes - 1))
        img = Image.fromarray(img_array.astype(np.uint8))
        img.save(new_file_path)

In [22]:
import os, torch, sys, subprocess, ml_collections
import numpy as np
from PIL import Image
from torchvision import transforms as T
from tqdm import tqdm
sys.path.append('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff')
from DCT_utils import *

# Ensure local datasets.py is imported, not HuggingFace's datasets package
import importlib.util
import sys
import os

datasets_path = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/datasets.py'
spec = importlib.util.spec_from_file_location("datasets", datasets_path)
datasets = importlib.util.module_from_spec(spec)
sys.modules["datasets"] = datasets
spec.loader.exec_module(datasets)

from datasets import get_dataset
from utils import DCT_to_greyscale
png_folder = '/bask/projects/c/chenhp-data-gen/yifansun/project/lidm_test/Data/ACDC/preprocessed/png/training/Image'

In [17]:
def d(**kwargs):
    """Helper of creating a config dict."""
    return ml_collections.ConfigDict(initial_dictionary=kwargs)

config = ml_collections.ConfigDict()

config.seed = 1234
config.pred = 'noise_pred'
config.dataset = d(
        name='acdc_cond',
        path=('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Image','/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Label'),
        resolution=96,
        tokens=144,  # number of tokens to the network
        low_freqs=16,  # B**2 - m
        block_sz=4,  # B
        Y_bound=[502.0],  # eta
        Y_std=[5.855, 3.48, 2.358, 1.471, 3.492, 2.733, 2.039, 1.269, 2.369, 2.047, 1.485, 0.998, 1.381, 1.166, 0.998, 0.999],
        Cb_std=[1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5],
        Cr_std=[1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5],
        SNR_scale=4.0,
        greyscale=True,  # use greyscale images
    )
sys.path.append('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff')
dataset = get_dataset(**config.dataset)

using eta [502.0] for training Image and Label seperately.


In [21]:
label = dataset.train[1]['label']
print(label.shape)

torch.Size([144, 64])


In [26]:
reverse_label = DCT_to_greyscale(sample=label,
                 tokens=config.dataset.tokens,
                 low_freqs=config.dataset.low_freqs,
                 block_sz=config.dataset.block_sz,
                 Y_bound=config.dataset.Y_bound,
                 resolution=config.dataset.resolution,
                 reverse_order=dataset.train.reverse_order_label)

from PIL import Image
Image.fromarray(reverse_label.astype(np.uint8)).save('reverse_label.png')

In [3]:
def convert_png_to_jpg(folder_path,newpath=None):
    """
    Converts all PNG images in the specified folder to JPG format.
    Args:
        folder_path (str): Path to the folder containing PNG images.
    """
    cnt = 0
    for subdir, dirs, files in os.walk(folder_path):
        for file in tqdm(files):
            if file.endswith(".png"):
                file_path = os.path.join(subdir, file)
                img = Image.open(file_path)
                rgb_img = img
                new_filename = file[:-4] + '.jpg'
                
                if newpath is not None:
                    new_file_path = os.path.join(newpath, new_filename)
                    if not os.path.exists(os.path.dirname(new_file_path)):
                        os.makedirs(os.path.dirname(new_file_path))
                else:
                    new_file_path = os.path.join(subdir, new_filename)
                    
                rgb_img.save(new_file_path, "JPEG")
                # os.remove(file_path)  # delete the original PNG file
                cnt += 1

    print(f"{cnt} imgs convertedfrom png to jpg in the folder {folder_path}")

png_folder = '/bask/projects/c/chenhp-data-gen/yifansun/project/lidm_test/Data/ACDC/preprocessed/png/training/Image'
dest_dir = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Image'
convert_png_to_jpg(png_folder, dest_dir)

100%|██████████| 1828/1828 [00:03<00:00, 518.99it/s]

1828 imgs convertedfrom png to jpg in the folder /bask/projects/c/chenhp-data-gen/yifansun/project/lidm_test/Data/ACDC/preprocessed/png/training/Image


In [4]:
img_pth = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Image/patient001_frame01_1.jpg'
img = Image.open(img_pth)
np_img = np.array(img)
print(np_img.shape)

(96, 96)


In [5]:
png_folder = '/bask/projects/c/chenhp-data-gen/yifansun/project/lidm_test/Data/ACDC/preprocessed/png/training/Label'
dest_dir = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Label'
convert_png_to_jpg(png_folder, dest_dir)

100%|██████████| 1828/1828 [00:08<00:00, 207.40it/s]

1828 imgs convertedfrom png to jpg in the folder /bask/projects/c/chenhp-data-gen/yifansun/project/lidm_test/Data/ACDC/preprocessed/png/training/Label


In [4]:
def convert_jpg_to_ycbcr(imgpt:str):
    assert imgpt.endswith('.jpg'), "Input image must be a JPG file."
    img = Image.open(imgpt).convert('RGB')
    img_ycbcr = img.convert('YCbCr')
    y, cb, cr = img_ycbcr.split()
    return np.array(y), np.array(cb), np.array(cr)

demo_img = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/JPGs/patient-1-frame1.jpg'
convert_jpg_to_ycbcr(demo_img)

(array([[ 36,  39,  43, ..., 185, 188, 186],
        [ 10,   9,  10, ..., 169, 179, 179],
        [ 20,  15,   8, ..., 146, 149, 138],
        ...,
        [ 58,  41,  19, ...,  87,  75,  70],
        [174, 147, 103, ...,  84,  74,  67],
        [172, 183, 189, ...,  84,  77,  71]], dtype=uint8),
 array([[128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128],
        ...,
        [128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128]], dtype=uint8),
 array([[128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128],
        ...,
        [128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128],
        [128, 128, 128, ..., 128, 128, 128]], dtype=uint8))

In [2]:
import os
len(os.listdir('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/JPGs/'))

25351

In [4]:
from PIL import Image
import numpy as np
import os

In [3]:
label_dir = '/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Label/patient001_frame01_1_label.jpg'
arr = np.array(Image.open(label_dir).convert('L'))
np.unique(arr)

array([0, 1, 2, 3, 4], dtype=uint8)

In [7]:
unique_labels = []
for i in os.listdir('/bask/projects/c/chenhp-data-gen/yifansun/project/lidm_test/Data/ACDC/preprocessed/png/training/Label'):
    label_dir = os.path.join('/bask/projects/c/chenhp-data-gen/yifansun/project/lidm_test/Data/ACDC/preprocessed/png/training/Label', i)
    arr = np.array(Image.open(label_dir).convert('L'))
    if unique_labels is not None:
        unique_labels.extend(np.unique(arr).tolist())
    else:
        unique_label = np.unique(arr).tolist()
        for label in unique_label:
            if label not in unique_labels:
                unique_labels.append(label)
print(unique_labels)

[0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 2, 3, 0, 2, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 

In [1]:
import matplotlib.pyplot as plt

plt.style.available

['Solarize_Light2',
 '_classic_test_patch',
 '_mpl-gallery',
 '_mpl-gallery-nogrid',
 'bmh',
 'classic',
 'dark_background',
 'fast',
 'fivethirtyeight',
 'ggplot',
 'grayscale',
 'seaborn-v0_8',
 'seaborn-v0_8-bright',
 'seaborn-v0_8-colorblind',
 'seaborn-v0_8-dark',
 'seaborn-v0_8-dark-palette',
 'seaborn-v0_8-darkgrid',
 'seaborn-v0_8-deep',
 'seaborn-v0_8-muted',
 'seaborn-v0_8-notebook',
 'seaborn-v0_8-paper',
 'seaborn-v0_8-pastel',
 'seaborn-v0_8-poster',
 'seaborn-v0_8-talk',
 'seaborn-v0_8-ticks',
 'seaborn-v0_8-white',
 'seaborn-v0_8-whitegrid',
 'tableau-colorblind10']

In [1]:
from PIL import Image
import numpy as np
import os
os.sys.path.append('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff')
from DCT_utils import *

block_sz = 8
img_folder = 'scratch/datasets/ACDC/Unlabeled/Wholeheart/25023_JPGs'
file_list = os.listdir(img_folder)
print(f"Total images found: {len(file_list)}")

# set example
img_path = os.path.join(img_folder, file_list[0])
img = Image.open(img_path).convert('RGB')
img = np.array(img)
print(f"Image shape: {img.shape}")
print(f"{img.max()}, {img.min() }")

R = img[:, :, 0]
G = img[:, :, 1]
B = img[:, :, 2]

img_y = 0.299 * R + 0.587 * G + 0.114 * B
img_cb = -0.168736 * R - 0.331264 * G + 0.5 * B + 128
img_cr = 0.5 * R - 0.418688 * G - 0.081312 * B + 128

print(f"Y channel shape: {img_y.shape}, max: {img_y.max()}, min: {img_y.min()}")

y_blocks = split_into_blocks(img_y, block_sz)  # Y component, (64, 64) --> (64, 8, 8)
dct_y_blocks = dct_transform(y_blocks)  # (64, 8, 8)
dct_y_blocks = dct_y_blocks.astype(np.float16)
Y_coe = dct_y_blocks.reshape(-1, block_sz*block_sz)
Y_coe.max(), Y_coe.min()

Total images found: 25023
Image shape: (96, 96, 3)
254, 0
Y channel shape: (96, 96), max: 253.99999999999997, min: 0.0


(np.float16(1626.0), np.float16(-471.5))

In [1]:
import numpy as np
import torch

sample = torch.randint(0,255,(144,4,16)).float()
sample.shape

torch.Size([144, 4, 16])

In [2]:
DCT_blocks = sample.numpy()
num_per_freq = 16 // 4
DCT_blocks_Pos = []
for i in range(4):
    freq_pos = DCT_blocks[:, :, i * num_per_freq:(i + 1) * num_per_freq]  # (tokens, 4, low_freqs//num_positional_tokens)
    freq_pos = freq_pos.reshape(-1, 4 * num_per_freq)  # (tokens, 4*low_freqs//num_positional_tokens)
    DCT_blocks_Pos.append(freq_pos)
DCT_blocks_Pos = np.concatenate(DCT_blocks_Pos, axis=0) # (tokens*num_positional_tokens, 4*low_freqs//num_positional_tokens)
DCT_blocks_Pos.shape

(576, 16)

In [7]:
reverse_order = []
for i in range(4):
    reverse_order.append(DCT_blocks_Pos[i*144:(i+1)*144, :])

reverse_order_ = []
for i in range(4):
    freq_pos = reverse_order[i].reshape(-1, 4, 4)  # (tokens, 4, low_freqs//num_positional_tokens)
    reverse_order_.append(freq_pos)
reverse_order_ = np.concatenate(reverse_order_, axis=-1)  # (tokens,
(reverse_order_ - sample.numpy()).sum()

np.float32(0.0)

In [5]:
sample.reshape(144, -1) - DCT_blocks_Pos.reshape(144, 4*16)

/tmp/ipykernel_4145935/3185702675.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  sample.reshape(144, -1) - DCT_blocks_Pos.reshape(144, 4*16)


tensor([[   0.,    0.,    0.,  ...,  128.,  110.,  -18.],
        [ 145., -213.,   99.,  ..., -103.,  157., -104.],
        [ 115.,   78.,  -12.,  ..., -234.,  -23.,   50.],
        ...,
        [ -46.,  143.,   90.,  ...,  141.,  167., -106.],
        [ 139.,  -32.,   54.,  ...,   -8.,   82.,  -50.],
        [-106.,   86.,    4.,  ...,    0.,    0.,    0.]])

In [1]:
import numpy as np
import torch, os, sys
sys.path.append('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/')
from datasets import DCT_FA_Customized

dataset = DCT_FA_Customized(
            data_property={'max':[774.0]},
            root_dir='/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Unlabeled/Wholeheart',
            img_sz=96,
            low_freqs=16,
            block_sz=4,
            normalization='Y_bound',)




Found 25023 images in /bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Unlabeled/Wholeheart
Using Y_bound normalization for tokens.
Using num_fa_length=2, num_fa_repeats=4 for frequency-aware tokenization.
Using Y_bound=774.0 for scaling


In [2]:
from DCT_utils import *

reverse_fn = dataset.reverse_ordering
denormalize_fn = dataset.denormalize

sample = dataset[0]
print(f'sample:{sample.shape}')

sample:torch.Size([1152, 8])


In [3]:
from utils import PositionalToken_to_greyscale
from PIL import Image
sample = reverse_fn(denormalize_fn(sample))
print(f'sample after denorm and reverse:{sample.shape}')
image = PositionalToken_to_greyscale(
    sample=sample.numpy(),
    img_sz=dataset.img_sz,
    low_freqs=dataset.low_freqs,
    block_sz=dataset.block_sz,

)
Image.fromarray(image.astype(np.uint8)).save('test.png')

sample after denorm and reverse:torch.Size([16, 576])


In [1]:
import numpy as np

# Your original function for context
def split_into_blocks(image, block_sz):
    """Splits an image into blocks in row-major order."""
    blocks = []
    for i in range(0, image.shape[0], block_sz):
        for j in range(0, image.shape[1], block_sz):
            blocks.append(image[i:i + block_sz, j:j + block_sz])
    return np.array(blocks)

def reorder_to_squares(blocks, num_blocks_y, num_blocks_x):
    """
    Reorders blocks from row-major to a "square-shell" order.
    
    The first r^2 blocks in the new list will form the r x r top-left
    square of blocks from the original image.
    
    Args:
        blocks (np.array): The input blocks in row-major order.
        num_blocks_y (int): The number of blocks in the y-direction (rows).
        num_blocks_x (int): The number of blocks in the x-direction (cols).

    Returns:
        tuple:
            - new_blocks (np.array): The reordered blocks.
            - forward_permutation (np.array): The permutation array used.
                                              (new_index -> original_index)
    """
    
    # This array will store the original (row-major) indices 
    # in the new "square-shell" order.
    forward_permutation = []
    
    # We iterate up to the largest dimension to form squares
    max_dim = max(num_blocks_y, num_blocks_x)
    
    for r in range(1, max_dim + 1):
        # r is the side length of the current square (e.g., 1, 2, 3...)
        # We are adding the "L-shaped" shell for square r.
        
        # 1. Add the new right column of the shell (from top to bottom)
        j = r - 1  # Column index of the new shell
        if j < num_blocks_x:
            for i in range(r): # Row index from 0 to r-1
                if i < num_blocks_y:
                    # Convert 2D block index (i, j) to 1D row-major index
                    original_flat_index = i * num_blocks_x + j
                    forward_permutation.append(original_flat_index)
                    
        # 2. Add the new bottom row of the shell (from left to right)
        #    (We skip the corner, as it was added in the column part)
        i = r - 1 # Row index of the new shell
        if i < num_blocks_y:
            for j in range(r - 1): # Col index from 0 to r-2 (skips corner)
                if j < num_blocks_x:
                    original_flat_index = i * num_blocks_x + j
                    forward_permutation.append(original_flat_index)

    # Convert to a NumPy array for indexing
    forward_permutation = np.array(forward_permutation)
    
    # Use the permutation array to reorder the blocks
    # This is called "fancy indexing"
    new_blocks = blocks[forward_permutation]
    
    return new_blocks, forward_permutation

def restore_original_order(new_blocks, forward_permutation):
    """
    Restores the original row-major block order from the "square-shell" order.
    
    Args:
        new_blocks (np.array): The reordered blocks.
        forward_permutation (np.array): The permutation array from the
                                        reorder_to_squares function.
    
    Returns:
        np.array: The blocks in their original row-major order.
    """
    
    # We need the inverse permutation, which maps:
    # original_index -> new_index
    # np.argsort() on the forward_permutation gives us exactly this.
    inverse_permutation = np.argsort(forward_permutation)
    
    # Use the inverse permutation to "un-shuffle" the new_blocks array
    # back to its original order.
    original_blocks = new_blocks[inverse_permutation]
    
    return original_blocks

In [4]:
# --- 0. Setup ---
# Create a 4x4 image with unique values for easy tracking
image = np.array([[ 0,  1,  2,  3,  4,  5,  6,  7],
                  [ 8,  9, 10, 11, 12, 13, 14, 15],
                  [ 16, 17, 18, 19, 20, 21, 22, 23],
                  [24, 25, 26, 27, 28, 29, 30, 31],
                  [32, 33, 34, 35, 36, 37, 38, 39],
                  [40, 41, 42, 43, 44, 45, 46, 47],
                  [48, 49, 50, 51, 52, 53, 54, 55],
                  [56, 57, 58, 59, 60, 61, 62, 63]])

block_sz = 2
num_blocks_y = image.shape[0] // block_sz  # 2
num_blocks_x = image.shape[1] // block_sz  # 2

# --- 1. Original Function ---
blocks = split_into_blocks(image, block_sz)
print("Original Row-Major Blocks (B_00, B_01, B_10, B_11):")
# B_00 (idx 0)
# B_01 (idx 1)
# B_10 (idx 2)
# B_11 (idx 3)
print(blocks)
print("-" * 20)

# --- 2. Reorder to Squares ---
new_blocks, perm = reorder_to_squares(blocks, num_blocks_y, num_blocks_x)

print(f"Forward Permutation (new -> old): {perm}")
print("Reordered 'Square-Shell' Blocks (B_00, B_01, B_11, B_10):")
# B_00 (idx 0 -> 0)
# B_01 (idx 1 -> 1)
# B_11 (idx 3 -> 2)
# B_10 (idx 2 -> 3)
print(new_blocks)
print("-" * 20)

# --- 3. Restore Original Order ---
restored_blocks = restore_original_order(new_blocks, perm)
print("Restored Row-Major Blocks:")
print(restored_blocks)
print("-" * 20)

# --- 4. Verification ---
print(f"Restoration successful: {np.array_equal(blocks, restored_blocks)}")

Original Row-Major Blocks (B_00, B_01, B_10, B_11):
[[[ 0  1]
  [ 8  9]]

 [[ 2  3]
  [10 11]]

 [[ 4  5]
  [12 13]]

 [[ 6  7]
  [14 15]]

 [[16 17]
  [24 25]]

 [[18 19]
  [26 27]]

 [[20 21]
  [28 29]]

 [[22 23]
  [30 31]]

 [[32 33]
  [40 41]]

 [[34 35]
  [42 43]]

 [[36 37]
  [44 45]]

 [[38 39]
  [46 47]]

 [[48 49]
  [56 57]]

 [[50 51]
  [58 59]]

 [[52 53]
  [60 61]]

 [[54 55]
  [62 63]]]
--------------------
Forward Permutation (new -> old): [ 0  1  5  4  2  6 10  8  9  3  7 11 15 12 13 14]
Reordered 'Square-Shell' Blocks (B_00, B_01, B_11, B_10):
[[[ 0  1]
  [ 8  9]]

 [[ 2  3]
  [10 11]]

 [[18 19]
  [26 27]]

 [[16 17]
  [24 25]]

 [[ 4  5]
  [12 13]]

 [[20 21]
  [28 29]]

 [[36 37]
  [44 45]]

 [[32 33]
  [40 41]]

 [[34 35]
  [42 43]]

 [[ 6  7]
  [14 15]]

 [[22 23]
  [30 31]]

 [[38 39]
  [46 47]]

 [[54 55]
  [62 63]]

 [[48 49]
  [56 57]]

 [[50 51]
  [58 59]]

 [[52 53]
  [60 61]]]
--------------------
Restored Row-Major Blocks:
[[[ 0  1]
  [ 8  9]]

 [[ 2  3]
 

In [4]:
import numpy as np

data = np.load('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/output/acdc_wholeheart_uncond_fa_ec_moe_greyscale_uvit_mid_4by4_l6r9/samples/expert_distribution_0_step25000.npy')
data

array('in_block_0', dtype='<U10')

In [1]:
import numpy as np
data = np.load('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/acdc_4by4_y.npy')
data.shape

(14413248, 16)

In [2]:
import os
datalist = os.listdir('/bask/projects/c/chenhp-data-gen/yifansun/project/DCTdiff/data/scratch/datasets/ACDC/Unlabeled/Wholeheart/cache_dct_fa_4by4_low12_l4r9_minmax')
for i in datalist:
    if not i.endswith('.pt'):
        print(i)